# EV ChargeOps — Demonstração do Protótipo Funcional

**FIAP Enterprise Challenge 2026 · GoodWe Energy Innovation Lab · Sprint 02**

Este notebook demonstra o funcionamento completo da plataforma:

1. Setup e criação do banco de dados
2. Simulação de 90 dias de sessões de recarga (10 unidades, 3 pontos GoodWe HCA G2)
3. Motor de Rateio RHPS — cálculo das faturas de junho/2026
4. Aura — Detecção de Anomalias (Isolation Forest)
5. Aura — Previsão de Demanda (Holt-Winters)
6. Aura — Boletim Energético Inteligente
7. Visualizações

> **Pré-requisitos:** `pip install -r requirements.txt` na raiz do repositório.  
> Execute as células em ordem. O banco SQLite `ev_chargeops.db` é criado no diretório corrente.


## 1. Setup e Criação do Banco de Dados

In [ ]:
import sys, os
# Garante que o pacote ev_chargeops seja encontrado quando o notebook roda na raiz do projeto
sys.path.insert(0, os.path.abspath(".."))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from datetime import datetime

# Configurações visuais
plt.style.use("dark_background")
sns.set_palette("cool")
pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.max_columns", 15)
pd.set_option("display.width", 120)

print("✓ Bibliotecas importadas")


In [ ]:
# Remove banco anterior (garante estado limpo na demonstração)
if os.path.exists("ev_chargeops.db"):
    os.remove("ev_chargeops.db")
    print("✓ Banco anterior removido")

from ev_chargeops.database import create_tables, SessionLocal
from ev_chargeops import models

create_tables()
print("✓ Tabelas criadas (SQLite → ev_chargeops.db)")


## 2. Simulação de Dados — 90 Dias de Operação

O simulador gera um cenário realista para um condomínio com 10 unidades e 3 pontos
de recarga GoodWe HCA G2 (11 kW, OCPP 1.6). Cinco sessões anômalas são injetadas
deliberadamente para validar a detecção da Aura.


In [ ]:
from ev_chargeops.simulador import popular_banco

db = SessionLocal()
resultado = popular_banco(db)

total_sessoes = db.query(models.SessaoRecarga).count()
total_eventos = db.query(models.EventoTelemetria).count()
total_unidades = db.query(models.Unidade).count()

print(f"✓ Simulação concluída")
print(f"   Unidades cadastradas : {total_unidades}")
print(f"   Pontos de recarga    : {db.query(models.PontoRecarga).count()}")
print(f"   Sessões geradas      : {total_sessoes}")
print(f"   Eventos de telemetria: {total_eventos}")


In [ ]:
# Visão geral das sessões simuladas por unidade
from sqlalchemy import func, extract

dados_resumo = (
    db.query(
        models.Unidade.identificacao.label("Unidade"),
        func.count(models.SessaoRecarga.id).label("Sessões"),
        func.round(func.sum(models.SessaoRecarga.kwh_final - models.SessaoRecarga.kwh_inicial), 2)
            .label("kWh Total"),
    )
    .join(models.Usuario, models.Usuario.unidade_id == models.Unidade.id)
    .join(models.SessaoRecarga, models.SessaoRecarga.usuario_id == models.Usuario.id)
    .filter(models.SessaoRecarga.kwh_final.isnot(None))
    .group_by(models.Unidade.identificacao)
    .order_by(models.Unidade.identificacao)
    .all()
)

df_resumo = pd.DataFrame(dados_resumo, columns=["Unidade", "Sessões", "kWh Total"])
df_resumo["kWh/Sessão"] = (df_resumo["kWh Total"] / df_resumo["Sessões"]).round(2)
print(df_resumo.to_string(index=False))
print(f"\nTotal geral: {df_resumo['Sessões'].sum()} sessões | {df_resumo['kWh Total'].sum():.2f} kWh")


In [ ]:
# Distribuição de sessões por hora do dia
sessoes_db = db.query(models.SessaoRecarga).filter(models.SessaoRecarga.kwh_final.isnot(None)).all()
horas = [s.inicio.hour for s in sessoes_db]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Perfil de Uso — 90 dias de operação simulada", color="white", fontsize=13, y=1.02)

# Histograma por hora
axes[0].hist(horas, bins=24, range=(0, 24), color="#38bdf8", edgecolor="#0f172a", alpha=0.85)
axes[0].set_title("Sessões por Hora do Dia", color="#94a3b8")
axes[0].set_xlabel("Hora", color="#94a3b8")
axes[0].set_ylabel("Quantidade de sessões", color="#94a3b8")
axes[0].set_facecolor("#1e293b")
axes[0].tick_params(colors="#64748b")

# kWh por unidade
kwh_unidade = df_resumo.set_index("Unidade")["kWh Total"]
axes[1].barh(kwh_unidade.index, kwh_unidade.values, color="#a78bfa", alpha=0.85)
axes[1].set_title("kWh Total por Unidade (90 dias)", color="#94a3b8")
axes[1].set_xlabel("kWh", color="#94a3b8")
axes[1].set_facecolor("#1e293b")
axes[1].tick_params(colors="#64748b")

fig.patch.set_facecolor("#0f172a")
plt.tight_layout()
plt.savefig("docs/diagrams/perfil_uso.png", dpi=120, bbox_inches="tight", facecolor="#0f172a")
plt.show()
print("✓ Gráfico salvo em docs/diagrams/perfil_uso.png")


## 3. Motor de Rateio — RHPS (Rateio Híbrido Proporcional por Sessão)

Calcula as faturas de **junho/2026** para todas as 10 unidades.

**Fórmula:**
```
Fatura(unidade) = Σ(kWh_sessão) × R$0,6878/kWh × 1,03  +  R$500 × fração_ideal
```


In [ ]:
from ev_chargeops.rateio import calcular_todas_as_faturas, resumo_faturamento

MES = "2026-06"

faturas = calcular_todas_as_faturas(db, MES)
print(f"✓ {len(faturas)} faturas calculadas para {MES}")


In [ ]:
# Tabela completa de faturamento
linhas = []
for f in sorted(faturas, key=lambda x: x.unidade.identificacao):
    kwh_mes = sum(item.kwh_atribuido for item in f.itens)
    linhas.append({
        "Unidade"         : f.unidade.identificacao,
        "Fração Ideal"    : f"{f.unidade.fracao_ideal*100:.1f}%",
        "Sessões"         : len(f.itens),
        "kWh"             : round(kwh_mes, 2),
        "Variável (R$)"   : f.componente_variavel,
        "Fixo (R$)"       : f.componente_fixo,
        "TOTAL (R$)"      : f.total,
    })

df_fat = pd.DataFrame(linhas)
print("=" * 78)
print(f"  FATURAMENTO MENSAL — {MES}  |  Tarifa: R$ 0,6878/kWh  |  Perda: 3%")
print("=" * 78)
print(df_fat.to_string(index=False))
print("-" * 78)
print(f"  TOTAL GERAL                                        "
      f"R$ {df_fat['TOTAL (R$)'].sum():.2f}")
print("=" * 78)


In [ ]:
# Resumo executivo do faturamento
resumo = resumo_faturamento(db, MES)
print("RESUMO EXECUTIVO")
print("-" * 40)
for k, v in resumo.items():
    print(f"  {k:<35} {v}")


In [ ]:
# Gráfico de composição das faturas (variável x fixo)
fig, ax = plt.subplots(figsize=(11, 4))
fig.patch.set_facecolor("#0f172a")
ax.set_facecolor("#1e293b")

unidades = df_fat["Unidade"]
var = df_fat["Variável (R$)"]
fixo = df_fat["Fixo (R$)"]

ax.bar(unidades, var, label="Componente Variável (kWh)", color="#38bdf8", alpha=0.9)
ax.bar(unidades, fixo, bottom=var, label="Componente Fixo (manutenção)", color="#a78bfa", alpha=0.9)
ax.set_title(f"Composição da Fatura por Unidade — {MES}", color="white", pad=12)
ax.set_ylabel("R$", color="#94a3b8")
ax.tick_params(colors="#64748b", axis="x", rotation=30)
ax.tick_params(colors="#64748b", axis="y")
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter("R$ %.0f"))
ax.legend(facecolor="#334155", labelcolor="white", fontsize=9)
plt.tight_layout()
plt.savefig("docs/diagrams/faturamento_junho.png", dpi=120, bbox_inches="tight", facecolor="#0f172a")
plt.show()


## 4. Aura — Detecção de Anomalias (Isolation Forest)

O modelo é treinado sobre todo o histórico de sessões. Sessões com padrão
anômalo são flaggeadas antes do fechamento da fatura.

**Features usadas:** `kwh_consumido`, `duracao_horas`, `potencia_media_kw`, `hora_inicio`


In [ ]:
from ev_chargeops.ia.anomalias import detectar_e_persistir

# Usa todas as sessões do histórico para treino
todas_sessoes = db.query(models.SessaoRecarga).filter(
    models.SessaoRecarga.kwh_final.isnot(None)
).all()

df_anomalias = detectar_e_persistir(db, todas_sessoes, limpar_anteriores=True)

print(f"✓ Modelo treinado sobre {len(todas_sessoes)} sessões")
print(f"✓ {len(df_anomalias)} sessões anômalas detectadas")


In [ ]:
if not df_anomalias.empty:
    print("=" * 85)
    print("  SESSÕES ANÔMALAS DETECTADAS")
    print("=" * 85)
    display_cols = ["id", "tipo", "kwh_consumido", "duracao_horas", "potencia_media_kw", "score_normalizado"]
    df_show = df_anomalias[display_cols].copy()
    df_show["tipo"] = df_show["tipo"].str.replace("_", " ")
    df_show.columns = ["ID (8 chars)", "Tipo", "kWh", "Duração (h)", "Pot. Média (kW)", "Score Aura"]
    df_show["ID (8 chars)"] = df_show["ID (8 chars)"].str[:8] + "…"
    print(df_show.to_string(index=False))
    print("-" * 85)
    print("  Todas marcadas como 'pendente' — disponíveis para revisão manual no painel.")


In [ ]:
# Scatter plot: kWh vs Duração com destaque para anômalas
fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor("#0f172a")
ax.set_facecolor("#1e293b")

kwh_all  = [s.kwh_consumido for s in todas_sessoes]
dur_all  = [s.duracao_horas for s in todas_sessoes]

ids_anomalos = set(df_anomalias["id"].tolist()) if not df_anomalias.empty else set()
cores = ["#f87171" if s.id in ids_anomalos else "#38bdf840" for s in todas_sessoes]
sizes = [80 if s.id in ids_anomalos else 18 for s in todas_sessoes]

ax.scatter(dur_all, kwh_all, c=cores, s=sizes, alpha=0.85, linewidths=0)

# Destaque manual das anômalas
if not df_anomalias.empty:
    ax.scatter(
        df_anomalias["duracao_horas"], df_anomalias["kwh_consumido"],
        c="#f87171", s=90, edgecolors="white", linewidths=0.8,
        label=f"Anômalas ({len(df_anomalias)})", zorder=5
    )

ax.set_title("Distribuição das Sessões — Detecção de Anomalias", color="white", pad=12)
ax.set_xlabel("Duração (horas)", color="#94a3b8")
ax.set_ylabel("kWh consumido", color="#94a3b8")
ax.tick_params(colors="#64748b")
ax.legend(facecolor="#334155", labelcolor="white")
plt.tight_layout()
plt.savefig("docs/diagrams/anomalias_scatter.png", dpi=120, bbox_inches="tight", facecolor="#0f172a")
plt.show()


## 5. Aura — Previsão de Demanda (Holt-Winters)

Modelo de suavização exponencial com sazonalidade semanal (período = 7 dias).
Projeção para os 30 dias seguintes ao histórico.


In [ ]:
from ev_chargeops.ia.previsao import prever_demanda, metricas_historicas

df_previsao = prever_demanda(db, todas_sessoes, horizonte_dias=30)
metricas = metricas_historicas(todas_sessoes)

print("✓ Modelo Holt-Winters treinado")
print(f"   kWh médio diário histórico : {metricas['kwh_medio_diario']:.2f} kWh")
print(f"   Dias com recarga           : {metricas['dias_com_recarga']}")
print(f"   Dias sem recarga           : {metricas['dias_sem_recarga']}")
print(f"\n   Projeção 30 dias (total)   : {df_previsao['kwh_previsto'].sum():.1f} kWh")
print(f"   Projeção diária média      : {df_previsao['kwh_previsto'].mean():.2f} kWh/dia")


In [ ]:
print("\nPrimeiros 10 dias projetados:")
print(df_previsao.head(10).to_string(index=False))


In [ ]:
# Gráfico de previsão com intervalo de confiança
fig, ax = plt.subplots(figsize=(12, 4))
fig.patch.set_facecolor("#0f172a")
ax.set_facecolor("#1e293b")

datas = pd.to_datetime(df_previsao["data"])
ax.plot(datas, df_previsao["kwh_previsto"], color="#a78bfa", linewidth=2, label="Previsão")
ax.fill_between(
    datas,
    df_previsao["intervalo_inferior"],
    df_previsao["intervalo_superior"],
    alpha=0.2, color="#a78bfa", label="Intervalo 95%"
)
ax.axhline(metricas["kwh_medio_diario"], color="#38bdf870", linestyle="--",
           linewidth=1, label=f"Média histórica ({metricas['kwh_medio_diario']:.1f} kWh/dia)")

ax.set_title("Previsão de Demanda — Próximos 30 dias (Holt-Winters)", color="white", pad=12)
ax.set_ylabel("kWh/dia", color="#94a3b8")
ax.tick_params(colors="#64748b")
ax.legend(facecolor="#334155", labelcolor="white", fontsize=9)
plt.tight_layout()
plt.savefig("docs/diagrams/previsao_demanda.png", dpi=120, bbox_inches="tight", facecolor="#0f172a")
plt.show()


## 6. Aura — Boletim Energético Inteligente

Geração automática do relatório mensal em linguagem natural.


In [ ]:
from ev_chargeops.ia.boletim import gerar_boletim

boletim = gerar_boletim(
    mes_referencia=MES,
    resumo_rateio=resumo,
    anomalias_df=df_anomalias,
    previsao_df=df_previsao,
    metricas_hist=metricas,
    nome_instalacao="Energy Innovation Lab — FIAP/GoodWe · Estacionamento L1"
)
print(boletim)


## 7. Validação dos Casos Excepcionais do Modelo RHPS

Conforme especificado na Sprint 01:
- **Caso 1**: Sessão interrompida
- **Caso 2**: Unidade sem recarga no mês
- **Caso 3**: Múltiplos cartões RFID por unidade


In [ ]:
from ev_chargeops.database import SessionLocal as SL
from ev_chargeops.models import (
    Unidade, Usuario, CartaoRFID, PontoRecarga, SessaoRecarga
)
from ev_chargeops.rateio import calcular_fatura_unidade, TARIFA_KWH_PADRAO

db2 = SL()

# --- Setup de teste isolado ---
ponto_teste = db2.query(PontoRecarga).first()

def nova_unidade(identificacao, fracao):
    u = Unidade(identificacao=identificacao, fracao_ideal=fracao)
    db2.add(u); db2.flush(); return u

def novo_usuario(unidade_id, nome):
    u = Usuario(unidade_id=unidade_id, nome=nome, papel="morador")
    db2.add(u); db2.flush(); return u

def novo_cartao(usuario_id, uid):
    c = CartaoRFID(usuario_id=usuario_id, uid_fisico=uid, status="ativo")
    db2.add(c); db2.flush(); return c

MES_TESTE = "2026-05"


In [ ]:
# ---------------------------------------------------------------
# CASO 1 — Sessão interrompida: usa kWh parcial, flag de incerteza
# ---------------------------------------------------------------
u1 = nova_unidade("Teste 201", 0.10)
usr1 = novo_usuario(u1.id, "Sessão Interrompida")
c1 = novo_cartao(usr1.id, "RFID_TESTE_201")

sessao_interrompida = SessaoRecarga(
    usuario_id=usr1.id, cartao_id=c1.id, ponto_id=ponto_teste.id,
    inicio=datetime(2026, 5, 10, 19, 0), fim=datetime(2026, 5, 10, 19, 45),
    kwh_inicial=2000.0, kwh_final=2004.2,   # 4.2 kWh parciais antes da queda
    status_encerramento="interrompida",
)
db2.add(sessao_interrompida); db2.flush()

fatura_caso1 = calcular_fatura_unidade(db2, u1.id, MES_TESTE)
print("CASO 1 — Sessão interrompida")
print(f"  kWh parcial medido : 4.20 kWh")
print(f"  Componente variável: R$ {fatura_caso1.componente_variavel:.2f}")
print(f"  Componente fixo    : R$ {fatura_caso1.componente_fixo:.2f}")
print(f"  Total fatura       : R$ {fatura_caso1.total:.2f}")
print(f"  Status sessão      : {sessao_interrompida.status_encerramento}")
print("  → kWh parcial contabilizado normalmente. Sessão disponível para auditoria.\n")


In [ ]:
# ---------------------------------------------------------------
# CASO 2 — Unidade sem recarga no mês: variável = 0, fixo normal
# ---------------------------------------------------------------
u2 = nova_unidade("Teste 202", 0.08)
novo_usuario(u2.id, "Sem Recarga no Mês")

fatura_caso2 = calcular_fatura_unidade(db2, u2.id, MES_TESTE)
print("CASO 2 — Unidade sem recarga no mês")
print(f"  Sessões no período  : 0")
print(f"  Componente variável : R$ {fatura_caso2.componente_variavel:.2f}  ← zero")
print(f"  Componente fixo     : R$ {fatura_caso2.componente_fixo:.2f}  ← cobrado normalmente")
print(f"  Total fatura        : R$ {fatura_caso2.total:.2f}")
print("  → Disponibilidade da infraestrutura é rateada independente do uso.\n")


In [ ]:
# ---------------------------------------------------------------
# CASO 3 — Dois cartões RFID (dois veículos) vinculados à mesma unidade
# ---------------------------------------------------------------
u3 = nova_unidade("Teste 203", 0.12)
usr3a = novo_usuario(u3.id, "Titular A")
usr3b = novo_usuario(u3.id, "Titular B")
c3a = novo_cartao(usr3a.id, "RFID_TESTE_203A")
c3b = novo_cartao(usr3b.id, "RFID_TESTE_203B")

# Veículo A carrega 8 kWh
db2.add(SessaoRecarga(
    usuario_id=usr3a.id, cartao_id=c3a.id, ponto_id=ponto_teste.id,
    inicio=datetime(2026, 5, 12, 20, 0), fim=datetime(2026, 5, 12, 22, 0),
    kwh_inicial=3000.0, kwh_final=3008.0, status_encerramento="concluida_normal",
))
# Veículo B carrega 11 kWh
db2.add(SessaoRecarga(
    usuario_id=usr3b.id, cartao_id=c3b.id, ponto_id=ponto_teste.id,
    inicio=datetime(2026, 5, 14, 18, 0), fim=datetime(2026, 5, 14, 21, 0),
    kwh_inicial=3008.0, kwh_final=3019.0, status_encerramento="concluida_normal",
))
db2.flush()

fatura_caso3 = calcular_fatura_unidade(db2, u3.id, MES_TESTE)
kwh_total_c3 = sum(i.kwh_atribuido for i in fatura_caso3.itens)
print("CASO 3 — Dois cartões RFID / dois veículos na mesma unidade")
print(f"  Sessões contabilizadas : {len(fatura_caso3.itens)} (Veículo A: 8 kWh + Veículo B: 11 kWh)")
print(f"  kWh total da unidade   : {kwh_total_c3:.2f} kWh")
print(f"  Componente variável    : R$ {fatura_caso3.componente_variavel:.2f}")
print(f"  Componente fixo        : R$ {fatura_caso3.componente_fixo:.2f}")
print(f"  Total fatura           : R$ {fatura_caso3.total:.2f}")
print("  → Consumo somado automaticamente. Fatura única para a unidade.")

db2.commit()
db2.close()


## 8. Conclusão — Evidência de Funcionamento

| Componente | Status | Evidência |
|---|---|---|
| Banco de dados (SQLite) | ✅ Funcional | Tabelas criadas, 10+ entidades persistidas |
| Simulador de dados | ✅ Funcional | 90 dias, 10 unidades, 3 pontos HCA G2, ~300 sessões |
| Motor de Rateio RHPS | ✅ Funcional | 10 faturas calculadas, 3 casos excepcionais validados |
| Aura — Detecção de Anomalias | ✅ Funcional | Isolation Forest, 5+ anomalias identificadas |
| Aura — Previsão de Demanda | ✅ Funcional | Holt-Winters, 30 dias projetados com IC |
| Aura — Boletim Energético | ✅ Funcional | Relatório em linguagem natural gerado automaticamente |
| API FastAPI (`backend/main.py`) | ✅ Pronta | Executável com `uvicorn backend.main:app --reload` |
| Dashboard (`frontend/index.html`) | ✅ Pronto | Abre no browser, consome a API local |

**Desvio documentado:** banco de dados usa SQLite no protótipo (planejado PostgreSQL + TimescaleDB).
A substituição requer alterar apenas a `DATABASE_URL` em `ev_chargeops/database.py`.
